# Lesson 9 — Toolbox: CLI, checkpoints and troubleshooting

⏱ **Duration:** ~15 min &nbsp;|&nbsp; 📚 **Builds on:** Lesson 8 &nbsp;|&nbsp; 💻 **Runs on:** a laptop CPU

**By the end of this lesson you will be able to:**

- use every `python -m myllm` command
- open a checkpoint and see what it contains
- run on a GPU, fix common errors, run the tests

> 🎤 *Teaching tip:* run the cells in order; every code cell is short enough to be explained live.

## 1. Command-line reference

| command | what it does |
|---------|--------------|
| `python -m myllm train [--preset tiny\|fast\|default] [--minutes N] [--data PATH] [--out FILE] [--seed N]` | tokenize, train, save the best checkpoint + a JSON timing log |
| `python -m myllm generate "prompt" [--length N] [--temperature T] [--top-k K] [--top-p P] [--seed S]` | sample text |
| `python -m myllm eval [--data PATH \| --text-file FILE]` | held-out (or custom) loss / perplexity / accuracy |
| `python -m myllm plot` | write `outputs/training.png` and `outputs/corpus.png` |
| `python -m myllm info` | print model size and training info |

Every command accepts `--model FILE` (default `models/model.pt`). Run `python -m myllm <command> -h` for all options.

In [1]:
import sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))  # lets us `import myllm` without installing it

import torch
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
torch.set_num_threads(4)
DATA = ROOT / "data" / "raw"       # training text lives here
MODELS = ROOT / "models"           # checkpoints are written here

In [2]:
# Notebook cells can run the CLI too (the leading ! runs a shell command)
CKPT = MODELS / "model.pt" if (MODELS / "model.pt").exists() else MODELS / "lesson_tiny.pt"
!cd {ROOT} && {sys.executable} -W ignore -m myllm info --model {CKPT}
!cd {ROOT} && {sys.executable} -W ignore -m myllm generate "the king" --model {CKPT} --length 25 --seed 3

/home/valerio/MyProjects/MyLLM/models/model.pt: 1,115,264 params, 3 layers, 4 heads, dim 128, context 64, vocab 4000
training info: {'steps': 2400, 'val_loss': 3.7906293272972107, 'preset': 'default', 'seed': 1337}


the king of the name of egypt: they said unto you, and do what he shall come and to him. 14: 16 and


## 2. What is inside a checkpoint?

In [3]:
ckpt = torch.load(CKPT, weights_only=True)
print("keys           :", list(ckpt))
print("model_config   :", ckpt["model_config"])
print("info           :", ckpt["info"])
print("vocabulary     :", len(ckpt["tokenizer"]["itos"]), "tokens, e.g.", ckpt["tokenizer"]["itos"][4:10])
n_tensors = len(ckpt["model_state"]); print(f"weights        : {n_tensors} tensors")
for name, w in list(ckpt["model_state"].items())[:6]:
    print(f"   {name:35s} {tuple(w.shape)}")

keys           : ['model_config', 'model_state', 'tokenizer', 'info']
model_config   : {'vocab_size': 4000, 'block_size': 64, 'n_layer': 3, 'n_head': 4, 'n_embd': 128, 'dropout': 0.1}
info           : {'steps': 2400, 'val_loss': 3.7906293272972107, 'preset': 'default', 'seed': 1337}
vocabulary     : 4000 tokens, e.g. [',', 'the', 'and', ':', 'of', '.']
weights        : 41 tensors
   tok_emb.weight                      (4000, 128)
   pos_emb.weight                      (64, 128)
   blocks.0.ln1.weight                 (128,)
   blocks.0.ln1.bias                   (128,)
   blocks.0.attn.qkv.weight            (384, 128)
   blocks.0.attn.qkv.bias              (384,)


A checkpoint is **self-contained**: weights + architecture + vocabulary + training info. That is why `generate`
needs no flags. It is saved to a `.tmp` file and renamed, so an interrupted run never leaves a corrupt checkpoint.

## 3. Using a GPU

The code is device-agnostic: `python -m myllm train --device cuda` (needs a CUDA build of PyTorch and a working driver).
Check availability:

In [4]:
print("cuda available:", torch.cuda.is_available())
print("threads used  :", torch.get_num_threads())

cuda available: False
threads used  : 4


## 4. Troubleshooting

| symptom | cause | fix |
|---------|-------|-----|
| `no .txt files in data/raw` | corpus missing | `python scripts/download_full_texts.py` or add your own `.txt` files |
| `no model at models/model.pt` | not trained yet | `python -m myllm train` |
| `corpus too small` | fewer tokens than needed for the context | use more text or a smaller preset |
| training much slower than expected | thread contention / background load | close other programs; try `torch.set_num_threads(4)` |
| `CUDA ... driver too old` warning | broken GPU setup | ignore: CPU is used by default |
| output is all lower-case with spaces around hyphens | word-level tokenizer | expected behaviour (lesson 2) |
| loss `nan` | learning rate too high | lower `--lr` |

## 5. Run the test-suite

The `tests/` folder checks every module (tokenizer, model causality, training, sampling, CLI…) on a tiny synthetic corpus in ~20 seconds:

In [5]:
!cd {ROOT} && {sys.executable} -m pytest -q 2>&1 | tail -3

.......................................................................  [100%]
71 passed in 16.30s


---
## ✅ Key takeaways

- The CLI covers the whole workflow: train, generate, eval, plot, info.
- Checkpoints bundle weights, config and vocabulary.
- `pytest` verifies the project in seconds — run it after any change.

## 🧪 Exercises

1. Use `--text-file` to evaluate the model on a text you wrote.
2. Train a model with `--seed 1` and `--seed 2` for 1 minute each. Are the logs identical? Why / why not?
3. Break a test on purpose (e.g. remove the causal flag in `model.py`). Which tests fail?

## ➡️ Next

**Lesson 10 — Wrap-up:** glossary, quiz and project ideas.